# FreshCart ELT on Google Colab: APIs → Snowflake → SQL

This notebook runs the full pipeline from Colab:

1. **Extract**: Python calls the APIs and saves the raw JSON.
2. **Load**: the raw files are uploaded to Snowflake's **bronze** tables (`PUT` + `COPY INTO`).
3. **Transform**: SQL inside Snowflake cleans, deduplicates and enriches into **silver**, then publishes **gold**.

**Before you start**, add your Snowflake login in Colab's **Secrets** panel (the 🔑 key icon in the left sidebar). Switch on *Notebook access* for each one.

| Secret name | Example | Needed? |
|---|---|---|
| `SNOWFLAKE_ACCOUNT` | `myorg-myaccount` (Snowsight → your name → Account → Copy account identifier) | yes |
| `SNOWFLAKE_USER` | `CHANDRA` | yes |
| `SNOWFLAKE_PASSWORD` | your password, **or** a programmatic access token | one of these two |
| `SNOWFLAKE_PRIVATE_KEY` | the full text of an `rsa_key.p8` key-pair file | one of these two |
| `SNOWFLAKE_ROLE` | `SYSADMIN` (default) | optional |

Colab can't open a browser login window, so SSO (`externalbrowser`) does not work here. If your account requires MFA and blocks plain password logins, use a key pair or a programmatic access token instead.

Run the cells from top to bottom.

## 1. Install libraries

In [ ]:
!pip install -q snowflake-connector-python requests python-dotenv

## 2. Create the project files
Each cell below writes one file into `/content/freshcart_elt`. Just run them; you can read them later.

In [ ]:
!mkdir -p /content/freshcart_elt/sql

In [ ]:
%%writefile /content/freshcart_elt/requirements.txt
snowflake-connector-python>=3.6
requests>=2.31
python-dotenv>=1.0

Writing /content/freshcart_elt/requirements.txt


In [ ]:

%%writefile /content/freshcart_elt/mock_data.py
"""Mock API responses: the 8 FreshCart orders from the course, split by the channel
they came from, plus CRM, FX and pincode responses. Used with --mock so the whole
pipeline runs end to end without any real API.

Note the two order APIs use DIFFERENT JSON shapes. That is normal in real life,
and it is why bronze stores raw JSON and silver unifies it with SQL.
"""

# Web export: flat JSON (rows 1, 3, 4, 5, 6, 8 of the course table)
WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# Mobile API: nested JSON, camelCase, day-first dates (rows 2 and 7)
MOBILE_ORDERS = [
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# CRM: a log of tier changes. The SQL turns this into an SCD Type 2 table.
CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Gold", "effective_from": "2026-08-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# Same shape as https://api.frankfurter.app/2026-09-01..2026-09-03?from=USD&to=INR
FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# Same shape as https://api.postalpincode.in/pincode/600041
_PIN = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),
        "400050": ("Mumbai", "Maharashtra")}


def pincode_response(pin):
    if pin not in _PIN:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = _PIN[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]


Overwriting /content/freshcart_elt/mock_data.py


In [ ]:
!cat /content/freshcart_elt/sql/40_gold_fct_orders.sql

-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to    

In [ ]:
!grep -n -i -A 15 -B 5 "tier_multiplier\|Silver\|Gold" /content/freshcart_elt/sql/00_setup.sql

9-
10-CREATE DATABASE IF NOT EXISTS FRESHCART;
11-USE DATABASE FRESHCART;
12-
13-CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
14:CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
15-CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
16-CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
17:CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
18-CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation
19-CREATE SCHEMA IF NOT EXISTS ops;         -- pipeline run log
20-
21--- Landing area inside Snowflake for the files Python uploads
22-CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
23-CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;
24-
25--- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
26-CREATE TABLE IF NOT EXISTS bronze.orders_raw (
27-  ingest_seq   NUMBER AUTOINCREME

In [ ]:
import os

print("Project files:", os.listdir('/content/freshcart_elt'))
print("SQL files:", os.listdir('/content/freshcart_elt/sql'))

Project files: ['mock_data.py', 'requirements.txt', 'sql']
SQL files: []


In [ ]:
!cat /content/freshcart_elt/run_pipeline.py

"""FreshCart ELT pipeline.

    python run_pipeline.py --setup --mock    # first time: create objects, run on sample data
    python run_pipeline.py --mock            # run again on sample data (idempotent)
    python run_pipeline.py                   # real APIs from .env
    python run_pipeline.py --dry-run --mock  # extract only, no Snowflake needed

Order of work (each step needs the previous one):
  1 EXTRACT  APIs -> local landing files (raw JSON)
  2 LOAD     landing files -> Snowflake bronze tables (PUT + COPY INTO)
  3 TRANSFORM in Snowflake with SQL:
       10 cleanse orders -> 20 deduplicate -> (look up new pincodes) ->
       30 customers SCD2, 31 FX daily, 32 pincode geo -> 40 enrich + publish gold
  4 CHECK    data-quality assertions, then print the gold totals
"""
import argparse
import logging
import os
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

import extract

SQL = Path(__fi

In [ ]:
# %load /content/freshcart_elt/run_pipeline.py
"""FreshCart ELT pipeline.

    python run_pipeline.py --setup --mock    # first time: create objects, run on sample data
    python run_pipeline.py --mock            # run again on sample data (idempotent)
    python run_pipeline.py                   # real APIs from .env
    python run_pipeline.py --dry-run --mock  # extract only, no Snowflake needed

Order of work (each step needs the previous one):
  1 EXTRACT  APIs -> local landing files (raw JSON)
  2 LOAD     landing files -> Snowflake bronze tables (PUT + COPY INTO)
  3 TRANSFORM in Snowflake with SQL:
       10 cleanse orders -> 20 deduplicate -> (look up new pincodes) ->
       30 customers SCD2, 31 FX daily, 32 pincode geo -> 40 enrich + publish gold
  4 CHECK    data-quality assertions, then print the gold totals
"""
import argparse
import logging
import os
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

import extract

SQL = Path(__file__).parent / "sql"
log = logging.getLogger("pipeline")


def parse_args():
    p = argparse.ArgumentParser(description="FreshCart ELT pipeline")
    p.add_argument("--setup", action="store_true", help="run sql/00_setup.sql first")
    p.add_argument("--mock", action="store_true", help="use built-in sample API responses")
    p.add_argument("--run-date", help="logical date YYYY-MM-DD (default: today)")
    p.add_argument("--dry-run", action="store_true", help="extract only; do not touch Snowflake")
    p.add_argument("--strict", action="store_true", help="fail (not warn) on quality warnings")
    return p.parse_args()


class Run:
    """Logs each step to the console and to ops.run_log, with timing."""
    def __init__(self, conn, run_id):
        self.conn, self.run_id = conn, run_id

    def record(self, step, status, detail=""):
        if self.conn is None:
            return
        with self.conn.cursor() as cur:
            cur.execute("INSERT INTO ops.run_log (run_id, step, status, detail) VALUES (%s, %s, %s, %s)",
                        (self.run_id, step, status, str(detail)[:4000]))

    def step(self, name, fn, *a, **kw):
        t0 = time.time()
        log.info("▶ %s", name)
        try:
            out = fn(*a, **kw)
        except Exception as e:
            self.record(name, "failed", repr(e))
            log.error("✖ %s failed: %s", name, e)
            raise
        secs = time.time() - t0
        self.record(name, "success", f"{secs:.1f}s; result={out!r}"[:4000])
        log.info("✔ %s (%.1fs)", name, secs)
        return out


def main():
    load_dotenv()
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)-9s %(message)s", datefmt="%H:%M:%S")

    run_date = date.fromisoformat(args.run_date) if args.run_date else date.today()
    run_id = f"{run_date:%Y%m%d}_{datetime.now(timezone.utc):%H%M%S}"
    mock = args.mock
    http = extract.http_session()
    fx_start = run_date - timedelta(days=int(os.getenv("FX_LOOKBACK_DAYS", "35")))
    log.info("run_id=%s run_date=%s mock=%s", run_id, run_date, mock)

    # ---------------- 1. EXTRACT (sources that don't depend on Snowflake) ----------------
    files = {
        "web":       extract.write_landing("web", extract.web_orders(http, run_date, mock), run_id, run_date),
        "mobile":    extract.write_landing("mobile", extract.mobile_orders(http, run_date, mock), run_id, run_date),
        "crm":       extract.write_landing("crm", extract.crm_tier_events(http, run_date, mock), run_id, run_date),
        "fx":        extract.write_landing("fx", extract.fx_rates(http, fx_start, run_date, mock), run_id, run_date),
    }
    if args.dry_run:
        log.info("dry run: landing files written, Snowflake not touched: %s", files)
        return

    import snowflake_io as sf                     # imported here so --dry-run needs no connector
    conn = sf.connect(bootstrap=args.setup)
    run = Run(None, run_id)
    try:
        if args.setup:
            run.step("setup", sf.run_sql_file, conn, SQL / "00_setup.sql")
        run.conn = conn                           # ops.run_log exists from here on
        vars_ = {"run_id": run_id, "run_date": run_date.isoformat()}

        # ---------------- 2. LOAD raw files into bronze ----------------
        stage = lambda src: f"{src}/{run_date.isoformat()}/{run_id}"
        run.step("load web orders",    sf.load_file, conn, files["web"],    "bronze.orders_raw",    stage("orders_web"))
        run.step("load mobile orders", sf.load_file, conn, files["mobile"], "bronze.orders_raw",    stage("orders_mobile"))
        run.step("load crm",           sf.load_file, conn, files["crm"],    "bronze.customers_raw", stage("crm"))
        run.step("load fx",            sf.load_file, conn, files["fx"],     "bronze.fx_raw",        stage("fx"))

        # ---------------- 3. TRANSFORM inside Snowflake ----------------
        run.step("10 cleanse orders",  sf.run_sql_file, conn, SQL / "10_silver_orders_clean.sql", vars_)
        run.step("20 deduplicate",     sf.run_sql_file, conn, SQL / "20_silver_orders_dedup.sql", vars_)

        # Pincodes depend on the orders: only look up valid ones we have never seen.
        pins = [r[0] for r in sf.query_all(conn, """
            SELECT DISTINCT o.pincode FROM silver.orders_dedup o
            WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]
        log.info("new pincodes to look up: %s", pins)
        pin_file = extract.write_landing("pincode", extract.pincodes(http, pins, mock), run_id, run_date)
        run.step("load pincodes",      sf.load_file, conn, pin_file, "bronze.pincode_raw", stage("pincode"))

        run.step("30 customers scd2",  sf.run_sql_file, conn, SQL / "30_dim_customers_scd2.sql", vars_)
        run.step("31 fx daily",        sf.run_sql_file, conn, SQL / "31_ref_fx_daily.sql", vars_)
        run.step("32 pincode geo",     sf.run_sql_file, conn, SQL / "32_ref_pincode_geo.sql", vars_)
        run.step("40 enrich + gold",   sf.run_sql_file, conn, SQL / "40_gold_fct_orders.sql", vars_)

        # ---------------- 4. CHECK ----------------
        run.step("quality checks", quality_checks, conn, args.strict)
        n, revenue, points = sf.query_one(conn,
            "SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
        log.info("GOLD: %s orders, revenue ₹%s, %s loyalty points", n, revenue, points)
    finally:
        conn.close()


def quality_checks(conn, strict):
    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
    unless --strict is given."""
    import snowflake_io as sf
    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
    checks = [
        ("no fan-out in enrichment (rows out - rows in)",
         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
         lambda v: v == 0, "fail"),
        ("one gold row per order_id (duplicates)",
         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
        ("orders with no customer match",
         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("orders with an amount but no FX rate",
         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
         lambda v: v == 0, "warn"),
        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        (f"quarantine rate (max {max_q:.0%})",
         "SELECT COALESCE(COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0), 0) FROM silver.orders_checked",
         lambda v: float(v) <= max_q, "warn"),
    ]
    problems = []
    for name, sql, ok, severity in checks:
        value = sf.query_one(conn, sql)[0]
        good = ok(value)
        log.info("   %-48s %-8s %s", name, value, "OK" if good else severity.upper())
        if not good and (severity == "fail" or strict):
            problems.append(f"{name} = {value}")
    if problems:
        raise RuntimeError("quality checks failed: " + "; ".join(problems))
    return "passed"


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        logging.getLogger("pipeline").error("pipeline failed: %s", exc)
        sys.exit(1)


In [ ]:
file_path = "/content/freshcart_elt/run_pipeline.py"

with open(file_path, "r") as f:
    code = f.read()

old = '''        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
'''

new = '''        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("future-dated orders in silver.orders_dedup",
         "SELECT COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup",
         lambda v: v == 0, "fail"),
'''

if old in code:
    code = code.replace(old, new, 1)
    with open(file_path, "w") as f:
        f.write(code)
    print("Sixth check added successfully!")
else:
    print("Could not find the expected code. No changes made.")

Sixth check added successfully!


In [ ]:
!cat /content/freshcart_elt/mock_data.py

"""Mock API responses: the 8 FreshCart orders from the course, split by the channel
they came from, plus CRM, FX and pincode responses. Used with --mock so the whole
pipeline runs end to end without any real API.

Note the two order APIs use DIFFERENT JSON shapes. That is normal in real life,
and it is why bronze stores raw JSON and silver unifies it with SQL.
"""

# Web export: flat JSON (rows 1, 3, 4, 5, 6, 8 of the course table)
WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_d

In [ ]:
file_path = "/content/freshcart_elt/mock_data.py"

with open(file_path, "r") as f:
    code = f.read()

start = code.index("WEB_ORDERS = [")
end = code.index("\n]", start)

new_order = '''
    {"order_id": "ORD-1007", "cust_email": "priya.r@gmail.com",
     "cust_name": "Priya R", "order_date": "2027-01-01",
     "amount": "500.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2027-01-01 10:00:00"},
'''

code = code[:end] + new_order + code[end:]

with open(file_path, "w") as f:
    f.write(code)

print("Temporary future-dated order ORD-1007 added!")

Temporary future-dated order ORD-1007 added!


In [ ]:
!python /content/freshcart_elt/run_pipeline.py --mock

15:32:53 pipeline  run_id=20261111_153253 run_date=2026-11-11 mock=True
15:32:53 extract   web: 7 records -> landing/web/2026-11-11/web__20261111_153253.json.gz
15:32:53 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_153253.json.gz
15:32:53 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_153253.json.gz
15:32:53 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_153253.json.gz
15:32:53 snowflake.connector Python connector starting v5.2.0
15:32:53 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:32:53 numexpr.utils NumExpr defaulting to 2 threads.
15:32:53 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:32:53 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:32:53 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:32:54 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

15:35:38 pipeline  run_id=20261111_153538 run_date=2026-11-11 mock=True
15:35:38 extract   web: 7 records -> landing/web/2026-11-11/web__20261111_153538.json.gz
15:35:38 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_153538.json.gz
15:35:38 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_153538.json.gz
15:35:38 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_153538.json.gz
15:35:38 snowflake.connector Python connector starting v5.2.0
15:35:38 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:35:38 numexpr.utils NumExpr defaulting to 2 threads.
15:35:38 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:35:38 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:35:38 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:35:38 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
file_path = "/content/freshcart_elt/mock_data.py"

with open(file_path, "r") as f:
    code = f.read()

start = code.index('    {"order_id": "ORD-1007"')
end = code.index('},', start) + 2

code = code[:start] + code[end:]

with open(file_path, "w") as f:
    f.write(code)

print("Temporary order ORD-1007 removed!")

Temporary order ORD-1007 removed!


In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

15:38:04 pipeline  run_id=20261111_153804 run_date=2026-11-11 mock=True
15:38:04 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_153804.json.gz
15:38:04 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_153804.json.gz
15:38:04 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_153804.json.gz
15:38:04 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_153804.json.gz
15:38:04 snowflake.connector Python connector starting v5.2.0
15:38:04 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:38:04 numexpr.utils NumExpr defaulting to 2 threads.
15:38:04 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:38:04 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:38:04 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:38:05 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!grep -n -A 20 -B 5 "orders_raw" /content/freshcart_elt/sql/00_setup.sql

21--- Landing area inside Snowflake for the files Python uploads
22-CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
23-CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;
24-
25--- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
26:CREATE TABLE IF NOT EXISTS bronze.orders_raw (
27-  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
28-  source       STRING,          -- 'web' or 'mobile'
29-  raw          VARIANT,         -- the API record exactly as received
30-  run_id       STRING,
31-  extracted_at TIMESTAMP_TZ,
32-  src_file     STRING,
33-  src_row      NUMBER,
34-  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
35-);
36-CREATE TABLE IF NOT EXISTS bronze.customers_raw (
37-  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
38-  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
39-  loaded_at TI

In [ ]:
import sys
sys.path.insert(0, "/content/freshcart_elt")

import snowflake_io as sf

conn = sf.connect()
rows = sf.query_all(conn, """
    SELECT source, raw:order_id::STRING AS order_id, run_id
    FROM FRESHCART.BRONZE.ORDERS_RAW
    WHERE raw:order_id::STRING = 'ORD-1007'
""")

for row in rows:
    print(row)

conn.close()

('web', 'ORD-1007', '20261111_153410')
('web', 'ORD-1007', '20261111_153538')


In [ ]:
import snowflake_io as sf

conn = sf.connect()
cur = conn.cursor()

cur.execute("""
    DELETE FROM FRESHCART.BRONZE.ORDERS_RAW
    WHERE raw:order_id::STRING = 'ORD-1007'
""")

print("Temporary records deleted:", cur.rowcount)

cur.close()
conn.close()

Temporary records deleted: 2


In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

15:43:59 pipeline  run_id=20261111_154359 run_date=2026-11-11 mock=True
15:43:59 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_154359.json.gz
15:43:59 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_154359.json.gz
15:43:59 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_154359.json.gz
15:43:59 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_154359.json.gz
15:43:59 snowflake.connector Python connector starting v5.2.0
15:43:59 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:43:59 numexpr.utils NumExpr defaulting to 2 threads.
15:43:59 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:43:59 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:43:59 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:43:59 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

15:34:10 pipeline  run_id=20261111_153410 run_date=2026-11-11 mock=True
15:34:10 extract   web: 7 records -> landing/web/2026-11-11/web__20261111_153410.json.gz
15:34:10 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_153410.json.gz
15:34:10 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_153410.json.gz
15:34:10 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_153410.json.gz
15:34:10 snowflake.connector Python connector starting v5.2.0
15:34:10 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:34:10 numexpr.utils NumExpr defaulting to 2 threads.
15:34:10 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:34:10 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:34:10 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:34:11 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!python /content/freshcart_elt/run_pipeline.py --mock

15:12:40 pipeline  run_id=20261111_151240 run_date=2026-11-11 mock=True
15:12:40 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_151240.json.gz
15:12:40 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_151240.json.gz
15:12:40 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_151240.json.gz
15:12:40 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_151240.json.gz
15:12:40 snowflake.connector Python connector starting v5.2.0
15:12:40 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:12:40 numexpr.utils NumExpr defaulting to 2 threads.
15:12:40 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:12:40 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:12:40 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:12:41 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

15:13:35 pipeline  run_id=20261111_151335 run_date=2026-11-11 mock=True
15:13:35 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_151335.json.gz
15:13:35 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_151335.json.gz
15:13:35 extract   crm: 5 records -> landing/crm/2026-11-11/crm__20261111_151335.json.gz
15:13:35 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_151335.json.gz
15:13:35 snowflake.connector Python connector starting v5.2.0
15:13:36 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
15:13:36 numexpr.utils NumExpr defaulting to 2 threads.
15:13:36 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
15:13:36 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
15:13:36 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
15:13:37 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
import sys
sys.path.insert(0, "/content/freshcart_elt")
import snowflake_io

conn = snowflake_io.connect()
cur = conn.cursor()

queries = {
    "CUST-02 history": """
        SELECT cust_id, tier, valid_from, valid_to
        FROM FRESHCART.DIM.CUSTOMERS_SCD2
        WHERE cust_id = 'CUST-02'
        ORDER BY valid_from
    """,
    "ORD-1002 details": """
        SELECT order_id, tier, multiplier, amount_inr, loyalty_points
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE order_id = 'ORD-1002'
    """,
    "Total gold loyalty points": """
        SELECT SUM(loyalty_points)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """,
    "Gold rows and published_at": """
        SELECT order_id, tier, loyalty_points, published_at
        FROM FRESHCART.GOLD.FCT_ORDERS
        ORDER BY published_at DESC
    """
}

for name, query in queries.items():
    print("\n", name)
    cur.execute(query)
    for row in cur.fetchall():
        print(row)

cur.close()
conn.close()


 CUST-02 history
('CUST-02', 'Silver', datetime.date(2024, 1, 15), datetime.date(2026, 8, 14))
('CUST-02', 'Gold', datetime.date(2026, 8, 15), datetime.date(9999, 12, 31))

 ORD-1002 details
('ORD-1002', 'Gold', 2, Decimal('3536.00'), 70)

 Total gold loyalty points
(161,)

 Gold rows and published_at
('ORD-1002', 'Gold', 70, datetime.datetime(2026, 11, 11, 7, 14, 5, 536000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1001', 'Silver', 12, datetime.datetime(2026, 11, 9, 5, 11, 8, 807000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1005', 'Gold', 16, datetime.datetime(2026, 11, 9, 5, 11, 8, 807000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1003', 'Platinum', 63, datetime.datetime(2026, 11, 9, 5, 11, 8, 807000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))


In [ ]:
!grep -n -A80 "def quality_checks" /content/freshcart_elt/run_pipeline.py

134:def quality_checks(conn, strict):
135-    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
136-    unless --strict is given."""
137-    import snowflake_io as sf
138-    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
139-    checks = [
140-        ("no fan-out in enrichment (rows out - rows in)",
141-         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
142-         lambda v: v == 0, "fail"),
143-        ("one gold row per order_id (duplicates)",
144-         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
145-        ("orders with no customer match",
146-         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
147-        ("orders with an amount but no FX rate",
148-         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
149-         lambda v: v == 0, "warn"),
150- 

In [ ]:
!python /content/freshcart_elt/run_pipeline.py --mock

14:36:59 pipeline  run_id=20261111_143659 run_date=2026-11-11 mock=True
14:36:59 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_143659.json.gz
14:36:59 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_143659.json.gz
14:36:59 extract   crm: 4 records -> landing/crm/2026-11-11/crm__20261111_143659.json.gz
14:36:59 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_143659.json.gz
14:36:59 snowflake.connector Python connector starting v5.2.0
14:37:00 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
14:37:00 numexpr.utils NumExpr defaulting to 2 threads.
14:37:00 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
14:37:00 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
14:37:00 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
14:37:01 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
14:37:01 pipeline  pipeline failed: '

In [ ]:
!cat /content/freshcart_elt/snowflake_io.py

"""LOAD: connect to Snowflake, upload landing files to an internal stage (PUT) and copy
them into the bronze tables (COPY INTO). Also runs the SQL transformation files."""
import logging
import os
from pathlib import Path

import snowflake.connector

log = logging.getLogger("snowflake")


def connect(bootstrap=False):
    """bootstrap=True connects without a database/warehouse (they may not exist yet
    before --setup has run; 00_setup.sql creates them and USEs them)."""
    kw = {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "role": os.getenv("SNOWFLAKE_ROLE"),
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
        "database": os.getenv("SNOWFLAKE_DATABASE"),
        "session_parameters": {"QUERY_TAG": "freshcart_elt"},
    }
    if bootstrap:
        kw.pop("warehouse"); kw.pop("database")
    if os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE"):          # key-pair auth (recommended)
        kw["private_key_file"] = os.envi

In [ ]:
import os

files = [
    "extract.py",
    "mock_data.py",
    "snowflake_io.py",
    "run_pipeline.py"
]

for file in files:
    print(file, ":", os.path.exists(f"/content/freshcart_elt/{file}"))

extract.py : True
mock_data.py : True
snowflake_io.py : True
run_pipeline.py : True


In [ ]:
from google.colab import userdata
import os

os.environ["SNOWFLAKE_ACCOUNT"] = userdata.get("SNOWFLAKE_ACCOUNT")
os.environ["SNOWFLAKE_USER"] = userdata.get("SNOWFLAKE_USER")
os.environ["SNOWFLAKE_PASSWORD"] = userdata.get("SNOWFLAKE_PASSWORD")

print("Snowflake credentials loaded successfully!")

Snowflake credentials loaded successfully!


In [ ]:
import os

sql_files = [
    "00_setup.sql",
    "10_silver_orders_clean.sql",
    "20_silver_orders_dedup.sql",
    "30_dim_customers_scd2.sql",
    "31_ref_fx_daily.sql",
    "32_ref_pincode_geo.sql",
    "40_gold_fct_orders.sql"
]

for file in sql_files:
    path = f"/content/freshcart_elt/sql/{file}"
    print(file, ":", os.path.exists(path))

00_setup.sql : True
10_silver_orders_clean.sql : True
20_silver_orders_dedup.sql : True
30_dim_customers_scd2.sql : True
31_ref_fx_daily.sql : True
32_ref_pincode_geo.sql : True
40_gold_fct_orders.sql : True


In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --dry-run --mock --run-date 2026-09-30

/content/freshcart_elt
13:08:58 pipeline  run_id=20260930_130858 run_date=2026-09-30 mock=True
13:08:58 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_130858.json.gz
13:08:58 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_130858.json.gz
13:08:58 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_130858.json.gz
13:08:58 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_130858.json.gz
13:08:58 pipeline  dry run: landing files written, Snowflake not touched: {'web': PosixPath('/content/freshcart_elt/landing/web/2026-09-30/web__20260930_130858.json.gz'), 'mobile': PosixPath('/content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_130858.json.gz'), 'crm': PosixPath('/content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_130858.json.gz'), 'fx': PosixPath('/content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_1308

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --mock --run-date 2026-09-30 --setup

/content/freshcart_elt
13:10:32 pipeline  run_id=20260930_131032 run_date=2026-09-30 mock=True
13:10:32 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_131032.json.gz
13:10:32 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_131032.json.gz
13:10:32 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_131032.json.gz
13:10:32 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_131032.json.gz
13:10:32 snowflake.connector Python connector starting v5.2.0
13:10:32 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
13:10:33 numexpr.utils NumExpr defaulting to 2 threads.
13:10:33 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
13:10:33 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
13:10:33 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
13:10:3

In [ ]:
%%writefile /content/freshcart_elt/extract.py
"""EXTRACT: call each API and save the responses unchanged as gzipped newline-delimited
JSON files in the local landing folder.

No cleaning happens here (that is the point of ELT). Each record is wrapped in a
small envelope so we always know where and when it came from:

    {"source": "web", "run_id": "...", "extracted_at": "...", "data": {<original record>}}
"""
import gzip
import json
import logging
import os
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import mock_data

log = logging.getLogger("extract")


# ---------------------------------------------------------------- HTTP helpers
def http_session():
    """A requests session that retries transient failures with exponential backoff
    (1 s, 2 s, 4 s, ...) and honours the server's Retry-After header on 429."""
    retry = Retry(total=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",), respect_retry_after_header=True)
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.headers["User-Agent"] = "freshcart-elt/1.0"
    return s


def _auth(token_env):
    token = os.getenv(token_env)
    return {"Authorization": f"Bearer {token}"} if token else {}


def paginate(session, url, params=None, headers=None, items_key="data", next_key="next",
             timeout=30, max_pages=10_000):
    """Yield records from an API that returns {"data": [...], "next": "<url or null>"}.
    Change items_key / next_key to match your API (e.g. "results" / "next_page_url").
    If your API uses page numbers instead, loop on params["page"] until a page is empty."""
    page_url, page_params = url, dict(params or {})
    for _ in range(max_pages):
        r = session.get(page_url, params=page_params, headers=headers, timeout=timeout)
        r.raise_for_status()
        body = r.json()
        items = body.get(items_key, []) if isinstance(body, dict) else body
        yield from items
        nxt = body.get(next_key) if isinstance(body, dict) else None
        if not nxt:
            return
        page_url, page_params = nxt, None      # the next link already carries the query
    raise RuntimeError(f"{url}: more than {max_pages} pages, stopping")


# ---------------------------------------------------------------- landing files
def write_landing(source, records, run_id, run_date):
    """Write records as one .json.gz file per source per run. Returns the path, or None."""
    records = list(records)
    if not records:
        log.info("%s: 0 records, nothing written", source)
        return None
    folder = Path(os.getenv("LANDING_DIR", "landing")) / source / run_date.isoformat()
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps({"source": source, "run_id": run_id, "extracted_at": now,
                                "data": rec}, ensure_ascii=False) + "\n")
    log.info("%s: %d records -> %s", source, len(records), path)
    return path


# ---------------------------------------------------------------- one function per API
def web_orders(session, run_date, mock=False):
    if mock:
        return mock_data.WEB_ORDERS
    # Incremental pull: everything changed since the day before the logical run date.
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["WEB_ORDERS_URL"], params={"updated_since": since},
                    headers=_auth("WEB_ORDERS_TOKEN"))


def mobile_orders(session, run_date, mock=False):
    if mock:
        return mock_data.MOBILE_ORDERS
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["MOBILE_ORDERS_URL"], params={"updatedSince": since},
                    headers=_auth("MOBILE_ORDERS_TOKEN"), items_key="orders", next_key="nextPage")


def crm_tier_events(session, run_date, mock=False):
    if mock:
        return mock_data.CRM_TIER_EVENTS
    return paginate(session, os.environ["CRM_TIER_EVENTS_URL"], headers=_auth("CRM_TOKEN"))


def fx_rates(session, start, end, mock=False):
    """Daily USD->INR (and any other FX_BASES) rates from the Frankfurter API (ECB data).
    One record per base currency, kept exactly as the API returns it."""
    if mock:
        return mock_data.FX_RESPONSES
    base_url = os.getenv("FX_URL", "https://api.frankfurter.app")
    out = []
    for base in [b.strip() for b in os.getenv("FX_BASES", "USD").split(",") if b.strip()]:
        r = session.get(f"{base_url}/{start.isoformat()}..{end.isoformat()}",
                        params={"from": base, "to": "INR"}, timeout=30)
        r.raise_for_status()
        out.append(r.json())
    return out


def pincodes(session, pins, mock=False):
    """India Post lookup, one call per pincode. Only pincodes we don't know yet are asked."""
    base_url = os.getenv("PINCODE_URL", "https://api.postalpincode.in/pincode")
    out = []
    for pin in pins:
        if mock:
            resp = mock_data.pincode_response(pin)
        else:
            r = session.get(f"{base_url}/{pin}", timeout=30)
            r.raise_for_status()
            resp = r.json()
            time.sleep(0.2)                    # be polite to a free public API
        out.append({"pincode": pin, "response": resp})
    return out

Writing /content/freshcart_elt/extract.py


In [ ]:
import os

print("mock_data.py exists:", os.path.exists("/content/freshcart_elt/mock_data.py"))

mock_data.py exists: False


In [ ]:
%%writefile /content/freshcart_elt/snowflake_io.py
"""LOAD: connect to Snowflake, upload landing files to an internal stage (PUT) and copy
them into the bronze tables (COPY INTO). Also runs the SQL transformation files."""
import logging
import os
from pathlib import Path

import snowflake.connector

log = logging.getLogger("snowflake")


def connect(bootstrap=False):
    """bootstrap=True connects without a database/warehouse (they may not exist yet
    before --setup has run; 00_setup.sql creates them and USEs them)."""
    kw = {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "role": os.getenv("SNOWFLAKE_ROLE"),
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
        "database": os.getenv("SNOWFLAKE_DATABASE"),
        "session_parameters": {"QUERY_TAG": "freshcart_elt"},
    }
    if bootstrap:
        kw.pop("warehouse"); kw.pop("database")
    if os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE"):          # key-pair auth (recommended)
        kw["private_key_file"] = os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]
        kw["private_key_file_pwd"] = os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE_PWD")
    elif os.getenv("SNOWFLAKE_AUTHENTICATOR"):           # e.g. externalbrowser (SSO)
        kw["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]
    else:
        kw["password"] = os.environ["SNOWFLAKE_PASSWORD"]
    return snowflake.connector.connect(**{k: v for k, v in kw.items() if v})


def run_sql_file(conn, path, variables=None):
    """Run every statement in a .sql file. Variables become session variables, so the SQL
    can use them as $run_id, $run_date."""
    text = Path(path).read_text(encoding="utf-8")
    prefix = "".join(f"SET {k} = '{v}';\n" for k, v in (variables or {}).items())
    for cur in conn.execute_string(prefix + text, remove_comments=True):
        cur.close()
    log.info("ran %s", Path(path).name)


def query_one(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchone()


def query_all(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchall()


def load_file(conn, local_path, table, stage_dir):
    """PUT one landing file to @bronze.landing/<stage_dir>/ then COPY it into <table>.

    COPY remembers which files it has loaded (for 64 days), so re-running this for the
    same file loads nothing the second time: the load step is idempotent."""
    if local_path is None:
        return 0
    uri = "file://" + Path(local_path).resolve().as_posix()
    with conn.cursor() as cur:
        cur.execute(f"PUT '{uri}' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE")
        cur.execute(f"""
            COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
              SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                     $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
              FROM @bronze.landing/{stage_dir}/
            )
            FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
            ON_ERROR = 'ABORT_STATEMENT'
        """)
        rows = cur.fetchall()
    # COPY returns one row per file: (file, status, rows_parsed, rows_loaded, ...),
    # or a single message row when there was nothing new to load.
    loaded = sum(int(r[3]) for r in rows if len(r) > 3 and str(r[1]).startswith(("LOADED", "PARTIALLY")))
    log.info("%s: %d rows copied into %s", Path(local_path).name, loaded, table)
    return loaded

Writing /content/freshcart_elt/snowflake_io.py


In [ ]:
import os
os.makedirs("/content/freshcart_elt", exist_ok=True)
print("Folder created successfully")

Folder created successfully


In [ ]:
%%writefile /content/freshcart_elt/run_pipeline.py
"""FreshCart ELT pipeline.

    python run_pipeline.py --setup --mock    # first time: create objects, run on sample data
    python run_pipeline.py --mock            # run again on sample data (idempotent)
    python run_pipeline.py                   # real APIs from .env
    python run_pipeline.py --dry-run --mock  # extract only, no Snowflake needed

Order of work (each step needs the previous one):
  1 EXTRACT  APIs -> local landing files (raw JSON)
  2 LOAD     landing files -> Snowflake bronze tables (PUT + COPY INTO)
  3 TRANSFORM in Snowflake with SQL:
       10 cleanse orders -> 20 deduplicate -> (look up new pincodes) ->
       30 customers SCD2, 31 FX daily, 32 pincode geo -> 40 enrich + publish gold
  4 CHECK    data-quality assertions, then print the gold totals
"""
import argparse
import logging
import os
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

import extract

SQL = Path(__file__).parent / "sql"
log = logging.getLogger("pipeline")


def parse_args():
    p = argparse.ArgumentParser(description="FreshCart ELT pipeline")
    p.add_argument("--setup", action="store_true", help="run sql/00_setup.sql first")
    p.add_argument("--mock", action="store_true", help="use built-in sample API responses")
    p.add_argument("--run-date", help="logical date YYYY-MM-DD (default: today)")
    p.add_argument("--dry-run", action="store_true", help="extract only; do not touch Snowflake")
    p.add_argument("--strict", action="store_true", help="fail (not warn) on quality warnings")
    return p.parse_args()


class Run:
    """Logs each step to the console and to ops.run_log, with timing."""
    def __init__(self, conn, run_id):
        self.conn, self.run_id = conn, run_id

    def record(self, step, status, detail=""):
        if self.conn is None:
            return
        with self.conn.cursor() as cur:
            cur.execute("INSERT INTO ops.run_log (run_id, step, status, detail) VALUES (%s, %s, %s, %s)",
                        (self.run_id, step, status, str(detail)[:4000]))

    def step(self, name, fn, *a, **kw):
        t0 = time.time()
        log.info("▶ %s", name)
        try:
            out = fn(*a, **kw)
        except Exception as e:
            self.record(name, "failed", repr(e))
            log.error("✖ %s failed: %s", name, e)
            raise
        secs = time.time() - t0
        self.record(name, "success", f"{secs:.1f}s; result={out!r}"[:4000])
        log.info("✔ %s (%.1fs)", name, secs)
        return out


def main():
    load_dotenv()
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)-9s %(message)s", datefmt="%H:%M:%S")

    run_date = date.fromisoformat(args.run_date) if args.run_date else date.today()
    run_id = f"{run_date:%Y%m%d}_{datetime.now(timezone.utc):%H%M%S}"
    mock = args.mock
    http = extract.http_session()
    fx_start = run_date - timedelta(days=int(os.getenv("FX_LOOKBACK_DAYS", "35")))
    log.info("run_id=%s run_date=%s mock=%s", run_id, run_date, mock)

    # ---------------- 1. EXTRACT (sources that don't depend on Snowflake) ----------------
    files = {
        "web":       extract.write_landing("web", extract.web_orders(http, run_date, mock), run_id, run_date),
        "mobile":    extract.write_landing("mobile", extract.mobile_orders(http, run_date, mock), run_id, run_date),
        "crm":       extract.write_landing("crm", extract.crm_tier_events(http, run_date, mock), run_id, run_date),
        "fx":        extract.write_landing("fx", extract.fx_rates(http, fx_start, run_date, mock), run_id, run_date),
    }
    if args.dry_run:
        log.info("dry run: landing files written, Snowflake not touched: %s", files)
        return

    import snowflake_io as sf                     # imported here so --dry-run needs no connector
    conn = sf.connect(bootstrap=args.setup)
    run = Run(None, run_id)
    try:
        if args.setup:
            run.step("setup", sf.run_sql_file, conn, SQL / "00_setup.sql")
        run.conn = conn                           # ops.run_log exists from here on
        vars_ = {"run_id": run_id, "run_date": run_date.isoformat()}

        # ---------------- 2. LOAD raw files into bronze ----------------
        stage = lambda src: f"{src}/{run_date.isoformat()}/{run_id}"
        run.step("load web orders",    sf.load_file, conn, files["web"],    "bronze.orders_raw",    stage("orders_web"))
        run.step("load mobile orders", sf.load_file, conn, files["mobile"], "bronze.orders_raw",    stage("orders_mobile"))
        run.step("load crm",           sf.load_file, conn, files["crm"],    "bronze.customers_raw", stage("crm"))
        run.step("load fx",            sf.load_file, conn, files["fx"],     "bronze.fx_raw",        stage("fx"))

        # ---------------- 3. TRANSFORM inside Snowflake ----------------
        run.step("10 cleanse orders",  sf.run_sql_file, conn, SQL / "10_silver_orders_clean.sql", vars_)
        run.step("20 deduplicate",     sf.run_sql_file, conn, SQL / "20_silver_orders_dedup.sql", vars_)

        # Pincodes depend on the orders: only look up valid ones we have never seen.
        pins = [r[0] for r in sf.query_all(conn, """
            SELECT DISTINCT o.pincode FROM silver.orders_dedup o
            WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]
        log.info("new pincodes to look up: %s", pins)
        pin_file = extract.write_landing("pincode", extract.pincodes(http, pins, mock), run_id, run_date)
        run.step("load pincodes",      sf.load_file, conn, pin_file, "bronze.pincode_raw", stage("pincode"))

        run.step("30 customers scd2",  sf.run_sql_file, conn, SQL / "30_dim_customers_scd2.sql", vars_)
        run.step("31 fx daily",        sf.run_sql_file, conn, SQL / "31_ref_fx_daily.sql", vars_)
        run.step("32 pincode geo",     sf.run_sql_file, conn, SQL / "32_ref_pincode_geo.sql", vars_)
        run.step("40 enrich + gold",   sf.run_sql_file, conn, SQL / "40_gold_fct_orders.sql", vars_)

        # ---------------- 4. CHECK ----------------
        run.step("quality checks", quality_checks, conn, args.strict)
        n, revenue, points = sf.query_one(conn,
            "SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
        log.info("GOLD: %s orders, revenue ₹%s, %s loyalty points", n, revenue, points)
    finally:
        conn.close()


def quality_checks(conn, strict):
    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
    unless --strict is given."""
    import snowflake_io as sf
    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
    checks = [
        ("no fan-out in enrichment (rows out - rows in)",
         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
         lambda v: v == 0, "fail"),
        ("one gold row per order_id (duplicates)",
         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
        ("orders with no customer match",
         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("orders with an amount but no FX rate",
         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
         lambda v: v == 0, "warn"),
        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        (f"quarantine rate (max {max_q:.0%})",
         "SELECT COALESCE(COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0), 0) FROM silver.orders_checked",
         lambda v: float(v) <= max_q, "warn"),
    ]
    problems = []
    for name, sql, ok, severity in checks:
        value = sf.query_one(conn, sql)[0]
        good = ok(value)
        log.info("   %-48s %-8s %s", name, value, "OK" if good else severity.upper())
        if not good and (severity == "fail" or strict):
            problems.append(f"{name} = {value}")
    if problems:
        raise RuntimeError("quality checks failed: " + "; ".join(problems))
    return "passed"


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        logging.getLogger("pipeline").error("pipeline failed: %s", exc)
        sys.exit(1)

Writing /content/freshcart_elt/run_pipeline.py


In [ ]:
import os

print("Project files:", os.listdir('/content/freshcart_elt'))
print("SQL files:", os.listdir('/content/freshcart_elt/sql'))

Project files: ['mock_data.py', 'requirements.txt', 'sql', 'snowflake_io.py', 'extract.py', 'run_pipeline.py']
SQL files: ['00_setup.sql', '32_ref_pincode_geo.sql', '40_gold_fct_orders.sql', '20_silver_orders_dedup.sql', '31_ref_fx_daily.sql', '30_dim_customers_scd2.sql', '10_silver_orders_clean.sql']


In [ ]:
!pip install -r /content/freshcart_elt/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 39.2 MB/s  0:00:00


In [ ]:
import os
os.makedirs("/content/freshcart_elt/sql", exist_ok=True)
print("SQL folder created successfully")

SQL folder created successfully


In [ ]:
!python /content/freshcart_elt/run_pipeline.py

14:32:18 pipeline  run_id=20261111_143218 run_date=2026-11-11 mock=False
14:32:18 pipeline  pipeline failed: 'WEB_ORDERS_URL'


In [ ]:
%%writefile /content/freshcart_elt/sql/00_setup.sql
-- =====================================================================
-- One-time setup. Run with:  python run_pipeline.py --setup --mock
-- Needs a role that can create warehouses and databases (e.g. SYSADMIN).
-- Safe to re-run: everything is IF NOT EXISTS, except the two seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation
CREATE SCHEMA IF NOT EXISTS ops;         -- pipeline run log

-- Landing area inside Snowflake for the files Python uploads
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
  source       STRING,          -- 'web' or 'mobile'
  raw          VARIANT,         -- the API record exactly as received
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ---------- REF: small hand-maintained seed tables ----------
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

-- Filled incrementally from the India Post API (see 32_ref_pincode_geo.sql)
CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- ---------- QUARANTINE, GOLD, OPS ----------
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);

CREATE TABLE IF NOT EXISTS ops.run_log (
  run_id STRING, step STRING, status STRING, detail STRING,
  logged_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

Writing /content/freshcart_elt/sql/00_setup.sql


In [ ]:
import os

print(os.path.exists('/content/freshcart_elt'))
print(os.path.exists('/content/freshcart_elt/sql'))

False
False


In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles and folders:")
print(os.listdir('/content'))

Current directory: /content

Files and folders:
['.config', 'sample_data']


In [ ]:

%%writefile /content/freshcart_elt/sql/10_silver_orders_clean.sql
-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT
    ingest_seq, source,
    TRIM(order_id)                                    AS order_id,
    LOWER(TRIM(cust_email))                           AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),          -- mobile API sends day first
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    )                                                 AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
    UPPER(TRIM(currency))                             AS currency,
    TRIM(pincode)                                     AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
  ) AS reasons
FROM typed;

-- Good rows
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being
-- quarantined again when a later run re-extracts it (same source, order, version).
INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:updated_at::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;


Writing /content/freshcart_elt/sql/10_silver_orders_clean.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/20_silver_orders_dedup.sql
-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order, so reruns give the same answer.
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;

Writing /content/freshcart_elt/sql/20_silver_orders_dedup.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/30_dim_customers_scd2.sql
-- =====================================================================
-- DIM: build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
  SELECT raw:cust_id::STRING                 AS cust_id,
         LOWER(TRIM(raw:email::STRING))      AS email,
         INITCAP(TRIM(raw:tier::STRING))     AS tier,
         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
         loaded_at, ingest_seq
  FROM bronze.customers_raw
  -- the same event may be extracted on many runs; keep the latest copy
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
                '9999-12-31'::DATE)                                       AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM events
WHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;

Writing /content/freshcart_elt/sql/30_dim_customers_scd2.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/31_ref_fx_daily.sql
-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
-- 1. Flatten the API's {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.
-- 2. Rates are not published on weekends/holidays: fill each missing day with the
--    previous published rate, so every order date finds a rate.
-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.
-- =====================================================================
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
  SELECT r.raw:base::STRING          AS currency,
         f.key::DATE                 AS rate_date,
         f.value:INR::NUMBER(12,4)   AS rate_to_inr
  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f
  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date
                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1
),
bounds AS (
  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1 FROM published
),
spine AS (                                   -- one row per calendar day d0..d1
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d
  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))
  QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
  SELECT c.currency, s.d AS rate_date,
         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
           PARTITION BY c.currency ORDER BY s.d
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,
         p.rate_to_inr IS NULL AS is_carried_forward
  FROM spine s
  CROSS JOIN (SELECT DISTINCT currency FROM published) c
  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d
)
SELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled
UNION ALL
SELECT 'INR', d, 1.0, FALSE FROM spine;

Writing /content/freshcart_elt/sql/31_ref_fx_daily.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/32_ref_pincode_geo.sql
-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes from
-- the ref.state_region seed table.
-- =====================================================================
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, sr.region
  FROM (
    SELECT raw:pincode::STRING                                  AS pincode,
           raw:response[0]:PostOffice[0]:District::STRING       AS city,
           raw:response[0]:PostOffice[0]:State::STRING          AS state,
           loaded_at, ingest_seq
    FROM bronze.pincode_raw
    WHERE raw:response[0]:Status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1
  ) p
  LEFT JOIN ref.state_region sr ON sr.state = p.state
) s
ON t.pincode = s.pincode
WHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,'') <> NVL(s.region,''))
  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT (pincode, city, state, region, updated_at)
       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());

Writing /content/freshcart_elt/sql/32_ref_pincode_geo.sql


In [ ]:
%%writefile /content/freshcart_elt/sql/40_gold_fct_orders.sql
-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to     -- tier ON the order date
LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
LEFT JOIN ref.fx_daily fx
       ON  fx.currency = o.currency AND fx.rate_date = o.order_date  -- rate OF the order date
LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;

MERGE INTO gold.fct_orders t
USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,
  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,
  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,
  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,
  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,
   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)
VALUES
  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,
   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,
   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());

Writing /content/freshcart_elt/sql/40_gold_fct_orders.sql


## 3. Read your Snowflake login from Colab Secrets
The values go into environment variables, which the pipeline script reads. Nothing is printed.

In [ ]:
import os
from google.colab import userdata

# ---- Snowflake login ----
os.environ["SNOWFLAKE_ACCOUNT"] = userdata.get("SNOWFLAKE_ACCOUNT")
os.environ["SNOWFLAKE_USER"] = userdata.get("SNOWFLAKE_USER")
os.environ["SNOWFLAKE_PASSWORD"] = userdata.get("SNOWFLAKE_PASSWORD")

os.environ["SNOWFLAKE_ROLE"] = "ACCOUNTADMIN"
os.environ["SNOWFLAKE_WAREHOUSE"] = "FRESHCART_WH"
os.environ["SNOWFLAKE_DATABASE"] = "FRESHCART"

# ---- Pipeline settings ----
os.environ["LANDING_DIR"] = "/content/freshcart_elt/landing"
os.environ["QUARANTINE_MAX_RATE"] = "0.05"

# ---- Check credentials ----
required = ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"]
missing = [k for k in required if not os.getenv(k)]

print("Missing:", missing) if missing else print("Login settings loaded successfully")

Login settings loaded successfully


In [ ]:
!python /content/freshcart_elt/run_pipeline.py --setup --mock

14:42:24 pipeline  run_id=20261111_144224 run_date=2026-11-11 mock=True
14:42:24 extract   web: 6 records -> landing/web/2026-11-11/web__20261111_144224.json.gz
14:42:24 extract   mobile: 2 records -> landing/mobile/2026-11-11/mobile__20261111_144224.json.gz
14:42:24 extract   crm: 4 records -> landing/crm/2026-11-11/crm__20261111_144224.json.gz
14:42:24 extract   fx: 1 records -> landing/fx/2026-11-11/fx__20261111_144224.json.gz
14:42:25 snowflake.connector Python connector starting v5.2.0
14:42:25 snowflake.connector._common.extras pyarrow is installed (version: 23.0.1)
14:42:25 numexpr.utils NumExpr defaulting to 2 threads.
14:42:25 snowflake.connector._common.extras pandas is installed (version: 2.2.3)
14:42:25 snowflake.connector._common.extras numpy is installed (version: 2.4.6)
14:42:25 snowflake.connector._common.extras tzlocal is installed (version: 5.4.5)
14:42:26 snowflake.connector._common.extras sqlalchemy is installed (version: 2.1.7)
/usr/local/lib/python3.13/dist-packag

In [ ]:
!cat /content/freshcart_elt/sql/00_setup.sql

-- =====================================================================
-- One-time setup. Run with:  python run_pipeline.py --setup --mock
-- Needs a role that can create warehouses and databases (e.g. SYSADMIN).
-- Safe to re-run: everything is IF NOT EXISTS, except the two seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- 

In [ ]:
import snowflake_io

conn = snowflake_io.connect()

cur = conn.cursor()
cur.execute("USE WAREHOUSE FRESHCART_WH")

cur.execute("""
    SELECT order_id, reasons
    FROM FRESHCART.SILVER.ORDERS_CHECKED
    WHERE order_id = 'ORD-1006'
""")

for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

ModuleNotFoundError: No module named 'snowflake_io'

In [ ]:
import sys
sys.path.insert(0, "/content/freshcart_elt")

import snowflake_io

conn = snowflake_io.connect()

cur = conn.cursor()
cur.execute("USE WAREHOUSE FRESHCART_WH")

cur.execute("""
    SELECT order_id, reasons
    FROM FRESHCART.SILVER.ORDERS_CHECKED
    WHERE order_id = 'ORD-1006'
""")

for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')


In [ ]:
!grep -n -A 20 -B 5 "ARRAY_CONSTRUCT_COMPACT" /content/freshcart_elt/sql/10_silver_orders_clean.sql

40-    TRIM(pincode)                                     AS pincode,
41-    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
42-  FROM unified
43-)
44-SELECT *,
45:  ARRAY_CONSTRUCT_COMPACT(
46-    IFF(order_id IS NULL, 'missing_order_id', NULL),
47-    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
48-    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
49-    IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL),
50-    IFF(order_date IS NULL, 'unparseable_date', NULL),
51-    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
52-    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
53-  ) AS reasons
54-FROM typed;
55-
56--- Good rows
57-CREATE OR REPLACE TABLE silver.orders_clean AS
58-SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
59-FROM silver.orders_checked
60-WHERE ARRAY_SI

In [ ]:
!sed -n '56,85p' /content/freshcart_elt/sql/10_silver_orders_clean.sql

-- Good rows
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being
-- quarantined again when a later run re-extracts it (same source, order, version).
INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:updated_at::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;


In [ ]:
cur = snowflake_io.connect().cursor()
cur.execute("""
    SELECT source, order_id, COUNT(*)
    FROM FRESHCART.QUARANTINE.ORDERS
    WHERE order_id = 'ORD-1006'
    GROUP BY source, order_id
""")
print(cur.fetchall())
cur.close()

[('web', 'ORD-1006', 2)]


True

In [ ]:
conn = snowflake_io.connect()
cur = conn.cursor()

cur.execute("""
    SELECT
        source,
        order_id,
        row_data:updated_at::TIMESTAMP_NTZ AS updated_at,
        reasons
    FROM FRESHCART.QUARANTINE.ORDERS
    WHERE order_id = 'ORD-1006'
    ORDER BY updated_at
""")

for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

('web', 'ORD-1006', None, '[\n  "bad_email",\n  "bad_pincode"\n]')
('web', 'ORD-1006', None, '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')


In [ ]:
conn = snowflake_io.connect()
cur = conn.cursor()

cur.execute("""
    SELECT order_id, run_id, reasons
    FROM FRESHCART.QUARANTINE.ORDERS
    WHERE order_id = 'ORD-1006'
""")

for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

('ORD-1006', '20260930_131032', '[\n  "bad_email",\n  "bad_pincode"\n]')
('ORD-1006', '20261111_144224', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')


In [ ]:
!grep -n -A 30 -B 5 "MOCK_CRM_TIER_EVENTS" /content/freshcart_elt/mock_data.py

In [ ]:
!grep -n -i -A 25 -B 5 "tier" /content/freshcart_elt/mock_data.py

36-    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
37-     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
38-     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
39-]
40-
41:# CRM: a log of tier changes. The SQL turns this into an SCD Type 2 table.
42:CRM_TIER_EVENTS = [
43:    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
44:    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
45:    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
46:    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
47-]
48-
49-# Same shape as https://api.frankfurter.app/2026-09-01..2026-09-03?from=USD&to=INR
50-FX_RESPONSES = [
51-    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "202

In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"]
)

print("Snowflake connection successful!")
conn.close()

/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Snowflake connection successful!


In [ ]:
!pip install snowflake-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 33.5 MB/s  0:00:00


## 4. Test the connection

In [ ]:
import sys, os, importlib

# Make the project folder importable, and reload the module in case an older copy is cached
sys.path.insert(0, "/content/freshcart_elt")
if not os.path.exists("/content/freshcart_elt/snowflake_io.py"):
    raise SystemExit("snowflake_io.py not found: run the step 2 cells (the %%writefile ones) first.")
import snowflake_io as sf
importlib.reload(sf)

print("Connecting to", os.environ.get("SNOWFLAKE_ACCOUNT"), "as", os.environ.get("SNOWFLAKE_USER"), "...")

try:
    conn = sf.connect(bootstrap=True)
    account, user, role, version = sf.query_one(
        conn, "SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_VERSION()")
    print(f" Connected!  account={account}  user={user}  role={role}  version={version}")
    conn.close()

except KeyError as e:
    print(f"Setting {e} is missing: run the login cell (step 3) first.")

except Exception as e:
    msg = str(e)
    print(" Connection failed:\n ", msg[:500])
    low = msg.lower()
    if "incorrect username or password" in low:
        print("→ Check the user name and password in step 3.")
    elif "mfa" in low or "multi-factor" in low:
        print("→ Your account needs MFA. In Snowflake: your name → Settings → Authentication →")
        print("  Programmatic access tokens → Generate. Put the token in SNOWFLAKE_PASSWORD in step 3.")
    elif "404" in low or "could not connect" in low or "account" in low or "certificate" in low:
        print("→ Account identifier is probably wrong. In Snowflake: your name (bottom left) →")
        print("  your account → 'Copy account identifier' (looks like ABCDEFG-XY12345),")
        print("  put it in SNOWFLAKE_ACCOUNT in step 3, re-run step 3, then this cell.")

Connecting to tt49522.ap-southeast-7.aws as ISHU2859 ...
 Connected!  account=TT49522  user=ISHU2859  role=ACCOUNTADMIN  version=10.40.103


## 5. Dry run: extract only, no Snowflake
Writes the raw sample API responses to `landing/`. Then we peek at one raw mobile record.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --dry-run --mock --run-date 2026-09-30
!zcat landing/mobile/*/*.json.gz | head -1

/content/freshcart_elt
Traceback (most recent call last):
  File "/content/freshcart_elt/run_pipeline.py", line 26, in <module>
    import extract
ModuleNotFoundError: No module named 'extract'
gzip: landing/mobile/*/*.json.gz: No such file or directory


## 6. Run the full pipeline on the sample data
`--setup` creates the warehouse, database, schemas and tables. It is only needed the first time; leaving it in is harmless.

Expected last line: **GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points**. One `WARN` for the quarantine rate (2 of 8 rows = 25%) is expected for this sample.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

/content/freshcart_elt
07:03:33 pipeline  run_id=20260930_070333 run_date=2026-09-30 mock=True
07:03:33 extract   web: 6 records -> /content/freshcart_elt/landing/web/2026-09-30/web__20260930_070333.json.gz
07:03:33 extract   mobile: 2 records -> /content/freshcart_elt/landing/mobile/2026-09-30/mobile__20260930_070333.json.gz
07:03:33 extract   crm: 4 records -> /content/freshcart_elt/landing/crm/2026-09-30/crm__20260930_070333.json.gz
07:03:33 extract   fx: 1 records -> /content/freshcart_elt/landing/fx/2026-09-30/fx__20260930_070333.json.gz
07:03:33 numexpr.utils NumExpr defaulting to 2 threads.
07:03:34 snowflake.connector.connection Snowflake Connector for Python Version: 4.8.0, Python Version: 3.13.15, Platform: Linux-6.6.122+-x86_64-with-glibc2.39
07:03:34 snowflake.connector.connection Connecting to GLOBAL Snowflake domain
07:03:35 pipeline  ▶ setup
07:03:44 snowflake ran 00_setup.sql
07:03:44 pipeline  ✔ setup (8.5s)
07:03:44 pipeline  ▶ load web orders
07:03:46 snowflake web__

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="SILVER"
)

cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM ORDERS_CLEAN")
print("Rows passing cleaning:", cur.fetchone()[0])

cur.close()
conn.close()

Rows passing cleaning: 6


In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="BRONZE"
)

cur = conn.cursor()
cur.execute("""
    SELECT *
    FROM QUARANTINE
""")

rows = cur.fetchall()

print("Quarantined rows:", len(rows))
for row in rows:
    print(row)

cur.close()
conn.close()

ProgrammingError: 002003 (42S02): SQL compilation error:
Object 'QUARANTINE' does not exist or not authorized. Your primary role ACCOUNTADMIN or one of your secondary roles must have at least one privilege granted on TABLE QUARANTINE. (request_id=464ec2b5-93b3-4e5d-8b11-01860faa15b8, sfqid=01c84bf1-000e-2aec-0003-0dfa00017ae2)

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="FRESHCART"
)

cur = conn.cursor()
cur.execute("SHOW TABLES IN SCHEMA FRESHCART.SILVER")

for row in cur.fetchall():
    print(row[1])

cur.close()
conn.close()

ORDERS_CHECKED
ORDERS_CLEAN
ORDERS_DEDUP
ORDERS_ENRICHED


In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="FRESHCART",
    schema="SILVER"
)

cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM ORDERS_DEDUP")
print("Rows after deduplication:", cur.fetchone()[0])

cur.close()
conn.close()

Rows after deduplication: 4


In [ ]:
cur = conn.cursor()
cur.execute("SHOW TABLES IN DATABASE FRESHCART")
for row in cur.fetchall():
    print(row[1], "| Schema:", row[3])
cur.close()

DatabaseError: 250002 (08003): Connection is closed.

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="FRESHCART"
)

cur = conn.cursor()
cur.execute("SHOW TABLES IN DATABASE FRESHCART")

for row in cur.fetchall():
    print(row[1], "| Schema:", row[3])

cur.close()
conn.close()

CUSTOMERS_RAW | Schema: BRONZE
FX_RAW | Schema: BRONZE
ORDERS_RAW | Schema: BRONZE
PINCODE_RAW | Schema: BRONZE
CUSTOMERS_SCD2 | Schema: DIM
FCT_ORDERS | Schema: GOLD
RUN_LOG | Schema: OPS
ORDERS | Schema: QUARANTINE
FX_DAILY | Schema: REF
PINCODE_GEO | Schema: REF
STATE_REGION | Schema: REF
TIER_MULTIPLIER | Schema: REF
ORDERS_CHECKED | Schema: SILVER
ORDERS_CLEAN | Schema: SILVER
ORDERS_DEDUP | Schema: SILVER
ORDERS_ENRICHED | Schema: SILVER


In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="FRESHCART",
    schema="QUARANTINE"
)

cur = conn.cursor()
cur.execute("SELECT * FROM ORDERS")

rows = cur.fetchall()
print("Quarantined rows:", len(rows))

for row in rows:
    print(row)

cur.close()
conn.close()

Quarantined rows: 2
('20260930_131032', datetime.datetime(2026, 11, 9, 5, 10, 54, 439000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), 6, 'web', 'ORD-1006', '[\n  "bad_email",\n  "bad_pincode"\n]', '{\n  "AMOUNT": 30,\n  "CURRENCY": "EUR",\n  "EMAIL": "vikram@@mail",\n  "INGEST_SEQ": 6,\n  "NAME": "Vikram",\n  "ORDER_DATE": "2026-09-04",\n  "ORDER_ID": "ORD-1006",\n  "PINCODE": "99999",\n  "REASONS": [\n    "bad_email",\n    "bad_pincode"\n  ],\n  "SOURCE": "web",\n  "UPDATED_AT": "2026-09-04 08:30:00.000"\n}')
('20260930_131032', datetime.datetime(2026, 11, 9, 5, 10, 54, 439000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), 4, 'web', 'ORD-1004', '[\n  "non_positive_amount"\n]', '{\n  "AMOUNT": -500,\n  "CURRENCY": "INR",\n  "EMAIL": "rahul.m@gmail.com",\n  "INGEST_SEQ": 4,\n  "NAME": "Rahul M",\n  "ORDER_DATE": "2026-09-03",\n  "ORDER_ID": "ORD-1004",\n  "PINCODE": "110001",\n  "REASONS": [\n    "non_positive_amount"\n  ],\n  "SOURCE": "web",

In [ ]:
from pathlib import Path

sql_dir = Path("/content/freshcart_elt/sql")

files = [
    "10_silver_orders_clean.sql",
    "20_silver_orders_dedup.sql",
    "40_gold_fct_orders.sql",
    "30_dim_customers_scd2.sql"
]

for filename in files:
    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)
    print((sql_dir / filename).read_text())


10_silver_orders_clean.sql
-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS

In [ ]:
import snowflake.connector
import os

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="FRESHCART"
)

cur = conn.cursor()

# 1. Actual raw records for ORD-1003
print("\n--- ORD-1003 RAW RECORDS ---")
cur.execute("""
SELECT source, ingest_seq, raw
FROM BRONZE.ORDERS_RAW
WHERE raw:order_id::STRING = 'ORD-1003'
   OR raw:orderId::STRING = 'ORD-1003'
ORDER BY source, ingest_seq
""")
for row in cur.fetchall():
    print(row)

# 2. Tier multipliers
print("\n--- TIER MULTIPLIERS ---")
cur.execute("SELECT * FROM REF.TIER_MULTIPLIER ORDER BY TIER")
for row in cur.fetchall():
    print(row)

# 3. Gold results for all three orders
print("\n--- GOLD RESULTS ---")
cur.execute("""
SELECT order_id, order_date, email, amount, currency,
       fx_rate, amount_inr, tier, multiplier, loyalty_points
FROM GOLD.FCT_ORDERS
WHERE order_id IN ('ORD-1003', 'ORD-1005', 'ORD-1001')
ORDER BY order_id
""")
for row in cur.fetchall():
    print(row)

# 4. Priya's customer history
print("\n--- PRIYA SCD2 HISTORY ---")
cur.execute("""
SELECT cust_id, email, tier, valid_from, valid_to, is_current
FROM DIM.CUSTOMERS_SCD2
WHERE email IN (
    SELECT DISTINCT LOWER(TRIM(raw:cust_email::STRING))
    FROM BRONZE.ORDERS_RAW
    WHERE raw:order_id::STRING IN ('ORD-1003', 'ORD-1005', 'ORD-1001')
      AND source = 'web'
)
ORDER BY email, valid_from
""")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()


--- ORD-1003 RAW RECORDS ---
('mobile', 8, '{\n  "customer": {\n    "email": "Meera@Outlook.com",\n    "name": "Meera S"\n  },\n  "deliveryPin": "400050",\n  "orderDate": "02/09/2026",\n  "orderId": "ORD-1003",\n  "total": {\n    "currency": "INR",\n    "value": "2,100.00"\n  },\n  "updatedAt": "2026-09-02T18:40:00"\n}')
('web', 2, '{\n  "amount": null,\n  "currency": "INR",\n  "cust_email": "meera@outlook.com",\n  "cust_name": "MEERA S",\n  "order_date": "2026-09-02",\n  "order_id": "ORD-1003",\n  "pincode": "400050",\n  "updated_at": "2026-09-02 09:10:00"\n}')

--- TIER MULTIPLIERS ---
('Gold', 2)
('Platinum', 3)
('Silver', 1)

--- GOLD RESULTS ---
('ORD-1001', datetime.date(2026, 9, 1), 'priya.r@gmail.com', Decimal('1250.00'), 'INR', Decimal('1.0000'), Decimal('1250.00'), 'Silver', 1, 12)
('ORD-1003', datetime.date(2026, 9, 2), 'meera@outlook.com', Decimal('2100.00'), 'INR', Decimal('1.0000'), Decimal('2100.00'), 'Platinum', 3, 63)
('ORD-1005', datetime.date(2026, 9, 3), 'priya.r@g

## 7. Look inside each layer
`show()` runs a query and returns a table.

In [ ]:
import pandas as pd
conn = sf.connect()

def show(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

show("SELECT ingest_seq, source, raw FROM bronze.orders_raw ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,RAW
0,1,web,"{\n ""amount"": ""1,250.00"",\n ""currency"": ""INR..."
1,2,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
2,3,web,"{\n ""amount"": ""42.50"",\n ""currency"": ""USD"",\..."
3,4,web,"{\n ""amount"": ""-500"",\n ""currency"": ""INR"",\n..."
4,5,web,"{\n ""amount"": ""₹ 899"",\n ""currency"": ""inr"",\..."
5,6,web,"{\n ""amount"": 30,\n ""currency"": ""EUR"",\n ""c..."
6,7,mobile,"{\n ""customer"": {\n ""email"": ""arjun.k@yaho..."
7,8,mobile,"{\n ""customer"": {\n ""email"": ""Meera@Outloo..."


### Silver: cleaned rows (6)

In [ ]:
show("SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, amount_missing FROM silver.orders_clean ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,ORDER_ID,EMAIL,NAME,ORDER_DATE,AMOUNT,CURRENCY,PINCODE,AMOUNT_MISSING
0,1,web,ORD-1001,priya.r@gmail.com,Priya R,2026-09-01,1250.00,INR,600041,False
1,2,web,ORD-1003,meera@outlook.com,Meera S,2026-09-02,None,INR,400050,True
2,3,web,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
3,5,web,ORD-1005,priya.r@gmail.com,Priya R,2026-09-03,899.00,INR,600041,False
4,7,mobile,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
5,8,mobile,ORD-1003,meera@outlook.com,Meera S,2026-09-02,2100.00,INR,400050,False


### Quarantine: rejected rows and why

In [ ]:
show("SELECT run_id, source, order_id, reasons FROM quarantine.orders")

,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20260930_070333,web,ORD-1006,"[\n ""bad_email"",\n ""bad_pincode""\n]"
1,20260930_070333,web,ORD-1004,"[\n ""non_positive_amount""\n]"


### Silver: one row per order (4)

In [ ]:
show("SELECT order_id, ingest_seq, updated_at, amount FROM silver.orders_dedup ORDER BY order_id")

,ORDER_ID,INGEST_SEQ,UPDATED_AT,AMOUNT
0,ORD-1001,1,2026-09-01 10:02:00,1250.00
1,ORD-1002,7,2026-09-01 11:15:00,42.50
2,ORD-1003,8,2026-09-02 18:40:00,2100.00
3,ORD-1005,5,2026-09-03 16:45:00,899.00


### Dim: customer tier history (SCD2)

In [ ]:
show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

,CUST_ID,EMAIL,TIER,VALID_FROM,VALID_TO,IS_CURRENT
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10,2026-09-02,False
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03,9999-12-31,True
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15,9999-12-31,True
3,CUST-03,meera@outlook.com,Platinum,2022-11-01,9999-12-31,True


### Ref: daily FX rates, weekends filled

In [ ]:
show("SELECT * FROM ref.fx_daily WHERE currency = 'USD' ORDER BY rate_date LIMIT 10")

,CURRENCY,RATE_DATE,RATE_TO_INR,IS_CARRIED_FORWARD
0,USD,2026-09-01,83.2000,False
1,USD,2026-09-02,83.3500,False
2,USD,2026-09-03,83.1000,False
3,USD,2026-09-04,83.1000,True
4,USD,2026-09-05,83.1000,True
5,USD,2026-09-06,83.1000,True
6,USD,2026-09-07,83.1000,True
7,USD,2026-09-08,83.1000,True
8,USD,2026-09-09,83.1000,True
9,USD,2026-09-10,83.1000,True


### Ref: pincode geography

In [ ]:
show("SELECT * FROM ref.pincode_geo")

,PINCODE,CITY,STATE,REGION,UPDATED_AT
0,600041,Chennai,Tamil Nadu,South,2026-10-01 00:04:06.456000-07:00
1,560034,Bengaluru,Karnataka,South,2026-10-01 00:04:06.456000-07:00
2,400050,Mumbai,Maharashtra,West,2026-10-01 00:04:06.456000-07:00


### Gold: the final table

In [ ]:
show("SELECT order_id, order_date, tier, amount, currency, fx_rate, amount_inr, loyalty_points, is_high_value, city, region FROM gold.fct_orders ORDER BY order_id")

,ORDER_ID,ORDER_DATE,TIER,AMOUNT,CURRENCY,FX_RATE,AMOUNT_INR,LOYALTY_POINTS,IS_HIGH_VALUE,CITY,REGION
0,ORD-1001,2026-09-01,Silver,1250.00,INR,1.0000,1250.00,12,False,Chennai,South
1,ORD-1002,2026-09-01,Silver,42.50,USD,83.2000,3536.00,35,True,Bengaluru,South
2,ORD-1003,2026-09-02,Platinum,2100.00,INR,1.0000,2100.00,63,True,Mumbai,West
3,ORD-1005,2026-09-03,Gold,899.00,INR,1.0000,899.00,16,False,Chennai,South


### Gold totals

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

,ORDERS,REVENUE_INR,POINTS
0,4,7785.00,126


### Run log

In [ ]:
show("SELECT step, status, detail, logged_at FROM ops.run_log ORDER BY logged_at DESC LIMIT 20")

,STEP,STATUS,DETAIL,LOGGED_AT
0,quality checks,success,2.0s; result='passed',2026-10-01 00:04:12.098000-07:00
1,40 enrich + gold,success,2.1s; result=None,2026-10-01 00:04:09.699000-07:00
2,32 pincode geo,success,1.2s; result=None,2026-10-01 00:04:07.147000-07:00
3,31 fx daily,success,1.4s; result=None,2026-10-01 00:04:04.450000-07:00
4,30 customers scd2,success,1.5s; result=None,2026-10-01 00:04:02.646000-07:00
5,load pincodes,success,2.2s; result=3,2026-10-01 00:04:00.624000-07:00
6,20 deduplicate,success,1.3s; result=None,2026-10-01 00:03:57.597000-07:00
7,10 cleanse orders,success,2.8s; result=None,2026-10-01 00:03:55.823000-07:00
8,load fx,success,1.4s; result=1,2026-10-01 00:03:52.560000-07:00
9,load crm,success,1.6s; result=4,2026-10-01 00:03:50.585000-07:00


## 8. Run it again: nothing should double
A second run extracts the same sample again, so bronze gets new copies. Deduplication and the gold `MERGE` absorb them: the totals stay 4 orders, ₹7,785.00, 126 points.

In [ ]:
!python run_pipeline.py --mock --run-date 2026-09-30
show("SELECT COUNT(*) AS bronze_rows FROM bronze.orders_raw")

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

## 9. Switch to your real APIs
Add these to Colab Secrets, run the cell, then run the pipeline **without** `--mock`.
Exchange rates (frankfurter.app) and pincodes (api.postalpincode.in) are free public APIs and need no key.

If your APIs page their results differently, edit `items_key` / `next_key` in `extract.py`. If their field names differ, edit only the `unified` block at the top of `sql/10_silver_orders_clean.sql`.

In [ ]:
for k in ["WEB_ORDERS_URL", "WEB_ORDERS_TOKEN", "MOBILE_ORDERS_URL", "MOBILE_ORDERS_TOKEN",
          "CRM_TIER_EVENTS_URL", "CRM_TOKEN"]:
    v = secret(k)
    if v:
        os.environ[k] = v
os.environ["FX_URL"] = "https://api.frankfurter.app"
os.environ["FX_BASES"] = "USD"
os.environ["PINCODE_URL"] = "https://api.postalpincode.in/pincode"

# !python run_pipeline.py          # uncomment to run on today's real data

## 10. Clean up (optional)
Removes everything this notebook created in Snowflake. Uncomment to run.

In [ ]:
# conn.cursor().execute("DROP DATABASE IF EXISTS FRESHCART")
# conn.cursor().execute("DROP WAREHOUSE IF EXISTS FRESHCART_WH")
conn.close()